# Practical 7: Data Wrangling for Aggregated Analysis on Balanced Cybersecurity Dataset
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Semester:** 5th Semester B.E. / B.Tech  
**Topic:** Aggregation, Temporal Resampling, Multi-Dimensional Pivot Tables, and Bot / Internal IP Filtering  
**Pipeline Input:** Practical 6 Balanced Training Dataset (`balanced_training_dataset.parquet` — 165,000 records)  

---

## 1. AIM & OBJECTIVES
### Aim
To aggregate and reshape balanced cybersecurity telemetry for deeper operational security insights by grouping attack frequencies per IP, resampling hourly and daily traffic, building multi-dimensional pivot tables, and filtering out automated bots and internal RFC 1918 traffic.

### Objectives
1. **Pipeline Continuity:** Ingest the 165,000-record balanced training dataset from Practical 6 (`balanced_training_dataset.parquet`).
2. **Client IP Threat Attribution:** Group by `client_ip` to identify persistent threat actors, attack volumes, and mean anomaly scores.
3. **Time-Series Resampling:** Resample datetime telemetry into hourly (`1h`) and daily (`1D`) intervals to monitor traffic velocity and attack surges.
4. **Cross-Tabulation Pivot Tables:** Construct pivot matrices cross-tabulating request types (GET, POST) and HTTP status codes against attack labels.
5. **Operational Noise Filtering:** Filter out automated crawlers (`user_agent`) and RFC 1918 private IPv4 ranges (`ipaddress.ip_address`), producing clean external operational telemetry.
6. **Diagnostic Visualizations:** Plot top attacking IPs, hourly time-series trends, pivot heatmaps, and filtering waterfall distributions.

---
## 2. SETUP & ENVIRONMENT
We import essential data science libraries, `ipaddress`, `pandas`, and `matplotlib`.

In [ ]:
import sys
import os
import time
import ipaddress
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Directory resolution
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORKSPACE_DIR = PRACTICAL_DIR.parent
P6_PROCESSED = WORKSPACE_DIR / 'Practical 6' / 'data' / 'processed'
PROCESSED_DIR = PRACTICAL_DIR / 'data' / 'processed'
AGGREGATED_DIR = PRACTICAL_DIR / 'outputs' / 'aggregated'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully.")

---
## 3. INGESTION & DATASET INTEGRITY AUDIT
We load `balanced_training_dataset.parquet` and convert timestamps.

In [ ]:
in_pq = P6_PROCESSED / 'balanced_training_dataset.parquet'
df = pd.read_parquet(in_pq)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], errors='coerce')

if 'is_attack' not in df.columns:
    df['is_attack'] = (df['label'] != 'benign').astype(np.int8)

print(f"Ingested Balanced Telemetry: {len(df):,} records, {len(df.columns)} columns")
print(f"Unique Client IPs: {df['client_ip'].nunique():,}")
print(f"Datetime Range: {df['datetime_utc'].min()} to {df['datetime_utc'].max()}")

---
## 4. GROUP BY CLIENT IP: ATTACK FREQUENCY & THREAT ACTORS
We aggregate total requests, attack events, and anomaly scores by client IP.

In [ ]:
ip_stats = df.groupby('client_ip').agg(
    total_requests=('line_number', 'count'),
    attack_count=('is_attack', 'sum'),
    benign_count=('is_attack', lambda s: int(np.sum(s == 0))),
    mean_anomaly_score=('anomaly_score', 'mean'),
    max_anomaly_score=('anomaly_score', 'max')
).reset_index()

ip_stats['attack_ratio'] = (ip_stats['attack_count'] / ip_stats['total_requests']).round(4)
ip_stats.sort_values(by=['attack_count', 'total_requests'], ascending=False, inplace=True)

top_attackers = ip_stats[ip_stats['attack_count'] > 0].head(10)
print("Top 10 Persistent Threat IPs:")
display(top_attackers)

---
## 5. TIME-SERIES RESAMPLING (HOURLY & DAILY INTERVALS)
We resample temporal telemetry into 1-hour and 1-day bins.

In [ ]:
df_ts = df.dropna(subset=['datetime_utc']).sort_values('datetime_utc').set_index('datetime_utc')

# Hourly aggregation
hourly_agg = df_ts.resample('1h').agg(
    total_requests=('line_number', 'count'),
    attacks=('is_attack', 'sum'),
    benign=('is_attack', lambda s: int(np.sum(s == 0))),
    mean_anomaly_score=('anomaly_score', 'mean')
).reset_index()

print(f"Hourly Resampled Intervals: {len(hourly_agg):,} bins")
display(hourly_agg.head(5))

---
## 6. MULTI-DIMENSIONAL PIVOT TABLES
Cross-tabulating request methods (GET, POST) and HTTP status codes against attack labels.

In [ ]:
req_col = 'request_method' if 'request_method' in df.columns else 'method'
pivot_req = pd.pivot_table(
    df, index=req_col, columns='label', values='line_number', aggfunc='count', fill_value=0
)
print("Request Method x Attack Label Pivot Table:")
display(pivot_req)

pivot_status = pd.pivot_table(
    df, index='status_code', columns='label', values='line_number', aggfunc='count', fill_value=0
)
print("Status Code x Attack Label Pivot Table:")
display(pivot_status)

---
## 7. BOT & RFC 1918 INTERNAL IP FILTERING
Filtering automated scrapers and internal IP ranges to produce clean operational telemetry.

In [ ]:
BOT_KEYWORDS = ['bot', 'crawler', 'spider', 'slurp', 'wget', 'curl', 'python-requests', 'nikto', 'sqlmap']

def is_bot_ua(ua):
    return any(k in str(ua).lower() for k in BOT_KEYWORDS)

def is_internal_ip(ip):
    try:
        ip_obj = ipaddress.ip_address(str(ip).strip())
        return ip_obj.is_private or ip_obj.is_loopback or ip_obj.is_link_local
    except ValueError:
        return False

bot_mask = df['user_agent'].apply(is_bot_ua)
internal_mask = df['client_ip'].apply(is_internal_ip)
clean_df = df[~(bot_mask | internal_mask)].copy().reset_index(drop=True)

print(f"Total Original Records: {len(df):,}")
print(f"Bots Filtered: {int(bot_mask.sum()):,} ({bot_mask.mean()*100:.2f}%)")
print(f"Internal IPs Filtered: {int(internal_mask.sum()):,} ({internal_mask.mean()*100:.2f}%)")
print(f"Clean Retained Operational Records: {len(clean_df):,} ({len(clean_df)/len(df)*100:.2f}%)")

# Export clean dataset
clean_df.to_parquet(PROCESSED_DIR / 'wrangled_filtered_dataset.parquet', index=False, compression='snappy')
print("Clean dataset exported to wrangled_filtered_dataset.parquet.")

---
## 8. PUBLICATION-QUALITY DIAGNOSTIC VISUALIZATIONS
Generating diagnostic figures for the wrangled and filtered data.

In [ ]:
# 1. Top 10 Attackers Bar Chart
plt.figure(figsize=(10, 4.5), dpi=150)
top10 = top_attackers.iloc[::-1]
plt.barh(top10['client_ip'], top10['attack_count'], color='#DC2626', edgecolor='#7F1D1D')
plt.title('Top 10 High-Frequency Malicious Client IPs', fontsize=12, fontweight='bold')
plt.xlabel('Attacks')
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 2. Hourly Traffic Resampling
plt.figure(figsize=(11, 4), dpi=150)
plt.plot(hourly_agg['datetime_utc'], hourly_agg['total_requests'], label='Total Traffic', color='#2563EB', linewidth=1.8)
plt.plot(hourly_agg['datetime_utc'], hourly_agg['attacks'], label='Attack Surges', color='#EF4444', linewidth=2.0)
plt.fill_between(hourly_agg['datetime_utc'], 0, hourly_agg['attacks'], color='#EF4444', alpha=0.25)
plt.title('Hourly Traffic Resampling (Total vs Attack Surge)', fontsize=12, fontweight='bold')
plt.xlabel('Time (UTC)')
plt.ylabel('Requests / Hour')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()